# Préparation ML (étape 2)

Entrée : `data/ml_data/processed/telco_clean.csv`  
(Telco encodé + features sentiment du Persona.)

Sortie : table **numérique**, prête pour logistic regression, Random Forest et XGBoost.

Ce notebook **ne ré-encode pas** Telco (déjà fait dans `data_preparing`).  
Il **ne scale pas** : le `StandardScaler` se fait **après** le split, et seulement pour la régression logistique.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "ml_data":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

SOURCE = PROJECT_ROOT / "data" / "ml_data" / "processed" / "telco_clean.csv"
OUTPUT = PROJECT_ROOT / "data" / "ml_data" / "processed" / "telco_train_ready.csv"

print(PROJECT_ROOT)
print(SOURCE.exists(), SOURCE)

c:\Users\wiame\Desktop\digital-twin-churn
True c:\Users\wiame\Desktop\digital-twin-churn\data\ml_data\processed\telco_clean.csv


In [2]:
df = pd.read_csv(SOURCE)
print(df.shape)
df.info()
df.head()

(7043, 59)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 59 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7043 non-null   int64  
 1   SeniorCitizen                          7043 non-null   int64  
 2   Partner                                7043 non-null   int64  
 3   Dependents                             7043 non-null   int64  
 4   tenure                                 7043 non-null   int64  
 5   PhoneService                           7043 non-null   int64  
 6   PaperlessBilling                       7043 non-null   int64  
 7   MonthlyCharges                         7043 non-null   float64
 8   TotalCharges                           7043 non-null   float64
 9   MultipleLines_No phone service         7043 non-null   bool   
 10  MultipleLines_Yes                      7043 non-null   bool   
 11  Inte

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,topic_payment_problem,topic_refund_request,topic_subscription_cancellation,topic_security_concern,topic_login_issue,topic_bug_report,topic_performance_issue,topic_data_sync_issue,topic_feature_request,topic_account_suspension
0,0,0,1,0,1,0,1,29.85,29.85,True,...,0,1,0,0,0,0,0,0,0,0
1,1,0,0,0,34,1,0,56.95,1889.50,False,...,0,0,0,0,0,0,0,1,0,0
2,1,0,0,0,2,1,1,53.85,108.15,False,...,0,0,0,0,1,0,0,0,0,0
3,1,0,0,0,45,0,0,42.30,1840.75,True,...,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,False,...,0,0,0,0,0,0,0,1,0,0


## Colonnes à retirer des features

| Colonne | Raison |
|---|---|
| `customerID` | identifiant, pas une variable prédictive |
| `sentiment` | texte ; déjà représenté par `sentiment_negative` / `_neutral` / `_positive` |
| `Churn` | **cible**, on la garde dans le fichier mais hors de `X` |

In [3]:
ID_COL = "customerID"
TARGET = "Churn"
TEXT_COLS = ["sentiment"]

missing = [c for c in [ID_COL, TARGET, *TEXT_COLS] if c not in df.columns]
if missing:
    raise ValueError(f"colonnes manquantes: {missing}")

print("NaN par colonne (non nuls seulement):")
nan_counts = df.isna().sum()
print(nan_counts[nan_counts > 0] if nan_counts.any() else "aucun NaN")

print("Churn unique:", sorted(df[TARGET].unique().tolist()))
print(df[TARGET].value_counts(normalize=True))

NaN par colonne (non nuls seulement):
aucun NaN
Churn unique: [0, 1]
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


In [4]:
ready = df.drop(columns=[ID_COL, *TEXT_COLS]).copy()

bool_cols = ready.select_dtypes(include="bool").columns
ready[bool_cols] = ready[bool_cols].astype(int)

object_cols = ready.select_dtypes(include=["object", "string"]).columns.tolist()
if object_cols:
    raise TypeError(
        "il reste des colonnes texte, à encoder ou à dropper: " + str(object_cols)
    )

if ready.isna().any().any():
    raise ValueError("NaN restants:\n" + str(ready.isna().sum()[ready.isna().any()]))

feature_cols = [c for c in ready.columns if c != TARGET]
ready = ready[feature_cols + [TARGET]]

print(ready.shape)
print(ready.dtypes.value_counts())
ready.head()

(7043, 57)
int64      52
float64     5
Name: count, dtype: int64


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,topic_refund_request,topic_subscription_cancellation,topic_security_concern,topic_login_issue,topic_bug_report,topic_performance_issue,topic_data_sync_issue,topic_feature_request,topic_account_suspension,Churn
0,0,0,1,0,1,0,1,29.85,29.85,1,...,1,0,0,0,0,0,0,0,0,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,0,0,0,0,1,0,0,0
2,1,0,0,0,2,1,1,53.85,108.15,0,...,0,0,0,1,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,1,...,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,0,...,0,0,0,0,0,0,1,0,0,1


In [5]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ready.to_csv(OUTPUT, index=False)
print("écrit:", OUTPUT)
print("features:", len(feature_cols))
print("cible:", TARGET)

écrit: c:\Users\wiame\Desktop\digital-twin-churn\data\ml_data\processed\telco_train_ready.csv
features: 56
cible: Churn


## `X` / `y` pour l’entraînement

Le split et le scaler (logreg uniquement) restent dans `train.py`, **après** cette table.

In [6]:
from sklearn.model_selection import train_test_split

X = ready.drop(columns=[TARGET])
y = ready[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print("taux churn train:", round(y_train.mean(), 4))
print("taux churn test :", round(y_test.mean(), 4))
assert list(X.columns) == feature_cols
assert TARGET not in X.columns

(5634, 56) (1409, 56)
taux churn train: 0.2654
taux churn test : 0.2654
